# Lectura y contratos de fuentes

Funciones compartidas por exploración y ETL. Homologan las columnas y las formas enteras de los identificadores, conservan ordinal de origen y verifican los duplicados contra el CSV completo. Se cargan mediante `%run` de Jupyter.

In [ ]:
from collections import Counter
import csv, gc, io, time, zipfile
import numpy as np
import pandas as pd
import json, re
from pathlib import Path

In [ ]:
def fuentes_viajes(raiz):
    manifest = json.loads((Path(raiz) / "catalogos/FUENTES.json").read_text(encoding="utf-8"))
    fuentes = {}
    for fuente in manifest["fuentes"]:
        ruta = Path(fuente["ruta"])
        coincidencia = re.fullmatch(r"recorridos_(\d{4})\.zip", ruta.name)
        if ruta.parent.as_posix() == "input" and coincidencia:
            ano = int(coincidencia[1])
            if ano in fuentes: raise ValueError(f"Más de una fuente anual registrada para {ano}.")
            fuentes[ano] = Path(raiz) / ruta
    if not fuentes: raise ValueError("El manifiesto no contiene fuentes anuales de recorridos.")
    return dict(sorted(fuentes.items()))


In [ ]:
def ids_enteros(serie):
    valor = serie.str.strip().str.removesuffix("BAEcobici").str.replace(r"\.0+$", "", regex=True)
    valor = valor.where(valor.str.fullmatch(r"\d+", na=False))
    return pd.to_numeric(valor, errors="coerce").fillna(-1).to_numpy(dtype="int64")

diccionarios = {"Modelo": {}, "Genero": {}}
def codificar(serie, campo):
    etiquetas = serie.fillna("Sin dato").str.strip().replace("", "Sin dato").str.upper()
    mapa = diccionarios[campo]
    for valor in etiquetas.unique():
        if valor not in mapa:
            mapa[valor] = len(mapa)
    return etiquetas.map(mapa).to_numpy(dtype=np.min_scalar_type(max(mapa.values())))

def variantes_estacion(bloque, ano, rol):
    campos = [f"id_estacion_{rol}", f"nombre_estacion_{rol}", f"long_estacion_{rol}", f"lat_estacion_{rol}"]
    tabla = bloque[campos].copy()
    tabla.columns = ["Estacion", "Nombre", "Lon", "Lat"]
    tabla["Estacion"] = ids_enteros(tabla.Estacion)
    for campo in ("Lon", "Lat"):
        tabla[campo] = pd.to_numeric(tabla[campo], errors="coerce").astype("float64").round(6)
    tabla["Nombre"] = tabla.Nombre.fillna("Sin dato")
    tabla["Ano"] = ano
    return tabla.groupby(["Ano","Estacion","Nombre","Lon","Lat"], dropna=False).size().rename("Eventos").reset_index()

In [ ]:
def cargar_viajes(RAIZ):
    global diccionarios
    diccionarios = {"Modelo": {}, "Genero": {}}
    fragmentos, variantes, faltantes, esquemas = [], [], [], []
    inicio_lectura = time.perf_counter()
    for ano, ruta in fuentes_viajes(RAIZ).items():
        total, nulos = 0, Counter()
        with zipfile.ZipFile(ruta) as archivo:
            miembros = [n for n in archivo.namelist() if n.lower().endswith(".csv")]
            if len(miembros) != 1: raise ValueError("El contrato de fuente exige un CSV por ZIP anual.")
            nombre = miembros[0]
            with archivo.open(nombre) as fuente:
                for numero, bruto in enumerate(pd.read_csv(fuente, dtype="string", chunksize=200_000), 1):
                    if numero == 1:
                        esquemas.append({"Ano": ano, "Columnas": len(bruto.columns), "CSV": nombre})
                    b = bruto.rename(columns={"Id_recorrido":"id_recorrido", "género":"genero"})
                    nulos.update(b.isna().sum().to_dict())
                    datos = {"Ano": np.full(len(b), ano, dtype="int16")}
                    for fuente_id, destino in [("id_recorrido","Viaje"),("id_usuario","Usuario"),
                                                ("id_estacion_origen","Origen"),("id_estacion_destino","Destino")]:
                        datos[destino] = ids_enteros(b[fuente_id])
                    datos["Inicio"] = pd.to_datetime(b.fecha_origen_recorrido, errors="coerce", format="mixed").to_numpy()
                    datos["Fin"] = pd.to_datetime(b.fecha_destino_recorrido, errors="coerce", format="mixed").to_numpy()
                    datos["Segundos"] = pd.to_numeric(b.duracion_recorrido.str.replace(",","",regex=False), errors="coerce").to_numpy(dtype=float, na_value=np.nan)
                    for fuente_geo, destino in [("long_estacion_origen","LonO"),("lat_estacion_origen","LatO"),
                                                 ("long_estacion_destino","LonD"),("lat_estacion_destino","LatD")]:
                        datos[destino] = pd.to_numeric(b[fuente_geo], errors="coerce").to_numpy(dtype=float, na_value=np.nan)
                    datos["Modelo"] = codificar(b.modelo_bicicleta,"Modelo")
                    datos["Genero"] = codificar(b.genero,"Genero")
                    datos["HuellaFila"] = pd.util.hash_pandas_object(b, index=False).to_numpy()
                    datos["FilaFuente"] = total + np.arange(1, len(b)+1, dtype="int32")
                    fragmentos.append(pd.DataFrame(datos))
                    variantes.extend(variantes_estacion(b, ano, rol) for rol in ("origen","destino"))
                    total += len(b)
                    if numero % 5 == 0:
                        print(f"{ano}: {total:,} registros leídos", flush=True)
        faltantes.extend({"Ano":ano,"Campo":k,"Nulos":int(v)} for k,v in nulos.items())
        print(f"{ano}: lectura completa, {total:,} registros", flush=True)
    viajes = pd.concat(fragmentos, ignore_index=True)
    variantes = pd.concat(variantes, ignore_index=True).groupby(["Ano","Estacion","Nombre","Lon","Lat"], dropna=False).Eventos.sum().reset_index()
    del fragmentos, bruto, b, datos
    gc.collect()
    print(f"Lectura terminada en {time.perf_counter()-inicio_lectura:.0f} s; memoria compacta: {viajes.memory_usage(deep=True).sum()/1e6:.0f} MB")
    return viajes, variantes, faltantes, esquemas, diccionarios

## Réplicas y conflictos

Una repetición con contenido idéntico conserva la primera ocurrencia. Los IDs con contenido distinto se identifican por separado para evitar una deduplicación arbitraria.

In [ ]:
def resolver_replicas(RAIZ, viajes):
    repetidos=viajes.duplicated(["Ano","Viaje"],keep=False)&viajes.Viaje.ge(0)
    replica=pd.Series(False,index=viajes.index)
    conflicto=pd.Series(False,index=viajes.index)
    for ano,grupo in viajes.loc[repetidos].groupby("Ano"):
        originales={i:[] for i in set(grupo.Viaje.astype(str))}
        with zipfile.ZipFile(RAIZ/f"input/recorridos_{ano}.zip") as z:
            nombre=next(x for x in z.namelist() if x.lower().endswith(".csv"))
            with z.open(nombre) as f:
                lector=csv.reader(io.TextIOWrapper(f,encoding="utf-8-sig",newline=""))
                next(lector)
                for fila in lector:
                    if not fila: continue
                    clave=fila[0].removesuffix("BAEcobici")
                    if clave in originales: originales[clave].append(tuple(fila))
        for clave,filas in originales.items():
            indices=grupo.index[grupo.Viaje.eq(int(clave))]
            if len(filas)!=len(indices): raise ValueError("No se pudo conciliar el ID repetido con la fuente.")
            if len(set(filas))==1: replica.loc[indices[1:]]=True
            else: conflicto.loc[indices]=True
    return ~replica&~conflicto&viajes.Viaje.ge(0),replica,conflicto
